# AI-Enhanced SIEM Framework — Validation Notebook

**Author:** Srujan Subramanya Attota  
**Dataset:** [GUIDE Dataset — Microsoft Security (Kaggle)](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction)  
**Purpose:** Validate the proposed AI-Enhanced SIEM Framework across four experiments:

| # | Experiment | What it measures |
|---|-----------|------------------|
| 1 | **Incident Detection Performance** | XGBoost classification accuracy on `IncidentGrade` labels |
| 2 | **Alert Prioritisation** | Hybrid risk score (ML + context) vs. ML-only for surfacing true positives |
| 3 | **Investigation Workload Reduction** | Incident-level grouping vs. alert-level review count |
| 4 | **Automated Response Recommendation** | Rule-based playbook mapping by category and incident grade |

---

## Prerequisites

Install required packages:

```bash
pip install pandas numpy scikit-learn xgboost matplotlib seaborn openpyxl
```

## Dataset Setup

1. Download `GUIDE_Train.csv` from the [Kaggle dataset page](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction).
2. Place it in the **same directory** as this notebook.
3. The notebook loads the first **300,000 rows** by default (adjust `nrows` in Cell 2 if needed).

## Outputs Generated

Running all cells produces the following files in the working directory:

| File | Type | Description |
|------|------|-------------|
| `table_detection_performance.csv` | CSV | Accuracy, Precision, Recall, F1 for the XGBoost model |
| `table_prioritisation_comparison.csv` | CSV | True Positive rate at Top 10/20/25/30% for both approaches |
| `table_investigation_workload.csv` | CSV | Alert rows vs. incident groups comparison |
| `table_response_summary.csv` | CSV | Response recommendations grouped by IncidentGrade |
| `table_response_action_distribution.csv` | CSV | Top 15 most-issued response actions |
| `SIEM_Framework_Results_Tables.xlsx` | Excel | All tables in a single workbook |
| `figure_prioritisation_comparison.png` | PNG | Line chart: Current AI-SIEM vs Proposed Framework |
| `figure_investigation_workload.png` | PNG | Bar chart: Review item count comparison |
| `figure_response_summary.png` | PNG | Bar chart: Records per IncidentGrade |
| `figure_normalised_confusion_matrix.png` | PNG | Heatmap: Per-class prediction accuracy (%) |

---
## Cell 1 — Environment Check

Confirms the working directory and lists available files. Run this first to verify the dataset CSV is present.

In [ ]:
import os
import pandas as pd

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

---
## Cell 2 — Load Dataset

Loads the first 300,000 rows of `GUIDE_Train.csv`.  
The GUIDE dataset contains Microsoft security telemetry with columns including `IncidentGrade` (target label), `Category`, `EvidenceRole`, `SuspicionLevel`, `ThreatFamily`, `MitreTechniques`, and `LastVerdict`.

**`IncidentGrade` values:**
- `TruePositive` — confirmed real incident
- `BenignPositive` — alert was correct but non-malicious
- `FalsePositive` — alert was incorrect

> Adjust `nrows` to load more data (full dataset = ~1M rows). Larger samples improve model accuracy but require more memory.

In [ ]:
import pandas as pd

df = pd.read_csv("GUIDE_Train.csv", nrows=300000)

print(df.shape)
print(df.columns.tolist())
df.head()

---
## Cell 3 — Class Distribution Check

Inspects how many records fall into each `IncidentGrade` category.  
This confirms whether the dataset is imbalanced before model training.

In [ ]:
df["IncidentGrade"].value_counts(dropna=False)

---
## Cell 4 — Feature Engineering and Train/Test Split

**Steps:**
1. Drop rows with missing `IncidentGrade`.
2. Remove identifier columns (`IncidentId`, `AlertId`, `OrgId`) that would cause data leakage.
3. Label-encode all categorical features and the target variable.
4. Fill missing values with `0`.
5. Split 80% train / 20% test with stratification to preserve class balance.

`raw_df` is retained as an unencoded copy for the investigation and response experiments in later cells.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Remove rows where IncidentGrade is missing
df = df.dropna(subset=["IncidentGrade"]).copy()

# Keep raw copy for investigation and response simulations
raw_df = df.copy()

target = "IncidentGrade"

# Drop ID columns so model does not learn identifiers
drop_cols = ["IncidentId", "AlertId", "OrgId"]

X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns], errors="ignore")
y = df[target]

# Encode target
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y.astype(str))

# Encode categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Fill missing values
X = X.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training data:", X_train.shape)
print("Testing data:",  X_test.shape)
print("Target classes:", target_encoder.classes_)

---
## Experiment 1 — Incident Detection Performance

### Cell 5 — Train XGBoost Classifier

Trains an **XGBoost** multi-class classifier to predict `IncidentGrade`.  
This represents the **Current AI-SIEM** detection capability.

**Model hyperparameters:**
- `n_estimators=200` — 200 boosting rounds
- `max_depth=6` — maximum tree depth
- `learning_rate=0.1` — step size shrinkage

**Output table:** Classification report + `current_results` dict (Accuracy, Macro Precision, Macro Recall, Macro F1).

> **Table saved as:** `table_detection_performance.csv`

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

current_ai_model = XGBClassifier(
    eval_metric="mlogloss",
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

current_ai_model.fit(X_train, y_train)

current_preds = current_ai_model.predict(X_test)
current_probs = current_ai_model.predict_proba(X_test)

print(classification_report(y_test, current_preds, target_names=target_encoder.classes_))

current_results = {
    "Approach": "Current AI-SIEM",
    "Accuracy": accuracy_score(y_test, current_preds),
    "Macro Precision": precision_score(y_test, current_preds, average="macro"),
    "Macro Recall":    recall_score(y_test, current_preds, average="macro"),
    "Macro F1":        f1_score(y_test, current_preds, average="macro")
}

current_results

### Cell 6 — Confusion Matrix (Figure 1)

Plots a **normalised confusion matrix** showing the percentage of each actual class correctly (or incorrectly) predicted.  
Row-normalised so each row sums to 100%.

> **Figure saved as:** `figure_normalised_confusion_matrix.png`

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, current_preds)

# Convert to percentages by row
cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_percent,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Normalised Confusion Matrix — Current AI-SIEM Model (%)")
plt.tight_layout()
plt.savefig("figure_normalised_confusion_matrix.png", dpi=300)
plt.show()
print("Saved: figure_normalised_confusion_matrix.png")

### Cell 7 — Per-Class Accuracy Table

Extracts the diagonal of the confusion matrix to show the correct classification rate for each `IncidentGrade` class individually.

In [ ]:
cm = confusion_matrix(y_test, current_preds)
classes = target_encoder.classes_

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

class_accuracy_table = pd.DataFrame({
    "Incident Grade": classes,
    "Correct Classification Rate": per_class_accuracy
})

class_accuracy_table

---
## Experiment 2 — Alert Prioritisation with Hybrid Risk Score

### Cell 8 — Extract TruePositive Probability

Extracts the model's predicted probability that each test record is a `TruePositive`.  
This is the **Current AI-SIEM** prioritisation signal — pure ML confidence.

In [ ]:
# Get index of TruePositive class in encoder
print("Encoded class order:", target_encoder.classes_)
tp_index = list(target_encoder.classes_).index("TruePositive")
print("TruePositive index:", tp_index)

# Probability that each row is TruePositive
tp_probability = current_probs[:, tp_index]

risk_df_tp = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y_test),
    "Predicted": target_encoder.inverse_transform(current_preds),
    "TruePositiveProbability": tp_probability
})

risk_df_tp.head()

### Cell 9 — Build Context-Aware Risk Score (Proposed Framework)

Constructs a **context-aware risk score** from raw feature columns using domain-weighted signals:

| Feature | Weight | Rationale |
|---------|--------|-----------|
| High-risk `Category` (e.g. Malware, Exfiltration) | 0.30 | Direct threat signal |
| `EvidenceRole` contains Impacted/Compromised | 0.20 | Asset impact signal |
| `SuspicionLevel` is populated | 0.15 | Analyst-generated suspicion |
| `ThreatFamily` is populated | 0.15 | Known threat family link |
| `MitreTechniques` is populated | 0.10 | ATT&CK technique coverage |
| `LastVerdict` contains Malicious/Suspicious | 0.10 | Previous verdict signal |

**Hybrid risk score formula:**  
`hybrid_score = 0.80 × TruePositive_probability + 0.20 × context_score`

The 80/20 weighting prioritises ML confidence while adding contextual uplift.

In [ ]:
# Align raw test data with encoded test indices
raw_test = raw_df.loc[X_test.index].copy()

context_score_v3 = pd.Series(0, index=raw_test.index, dtype=float)

# High-risk categories based on common security incident types
high_risk_categories = [
    "CommandAndControl", "Exfiltration", "CredentialAccess",
    "Malware", "Persistence", "PrivilegeEscalation",
    "Execution", "LateralMovement", "Impact"
]

if "Category" in raw_test.columns:
    context_score_v3 += raw_test["Category"].isin(high_risk_categories).astype(int) * 0.30

if "EvidenceRole" in raw_test.columns:
    context_score_v3 += raw_test["EvidenceRole"].astype(str).str.contains(
        "Impacted|Compromised|Related", case=False, na=False
    ).astype(int) * 0.20

if "SuspicionLevel" in raw_test.columns:
    context_score_v3 += raw_test["SuspicionLevel"].notna().astype(int) * 0.15

if "ThreatFamily" in raw_test.columns:
    context_score_v3 += raw_test["ThreatFamily"].notna().astype(int) * 0.15

if "MitreTechniques" in raw_test.columns:
    context_score_v3 += raw_test["MitreTechniques"].notna().astype(int) * 0.10

if "LastVerdict" in raw_test.columns:
    context_score_v3 += raw_test["LastVerdict"].astype(str).str.contains(
        "Malicious|Suspicious", case=False, na=False
    ).astype(int) * 0.10

# Normalise context score to [0, 1]
context_score_v3 = (context_score_v3 - context_score_v3.min()) / (
    context_score_v3.max() - context_score_v3.min() + 1e-9
)

# Proposed framework hybrid score
hybrid_risk_score_v3 = (0.80 * tp_probability) + (0.20 * context_score_v3.values)

risk_df_v3 = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y_test),
    "Predicted": target_encoder.inverse_transform(current_preds),
    "TruePositiveProbability": tp_probability,
    "ContextScore": context_score_v3.values,
    "HybridRiskScore": hybrid_risk_score_v3
})

risk_df_v3.head()

### Cell 10 — Prioritisation Comparison at Multiple Thresholds

Evaluates both approaches (Current AI-SIEM vs Proposed Framework) at Top 10%, 20%, 25%, and 30% of ranked alerts.  
Measures the **True Positive Rate** within each top group — how many of the highest-ranked alerts are genuine incidents.

Higher rates mean analysts spend less time on false alarms.

> **Table saved as:** `table_prioritisation_comparison.csv`

In [ ]:
rows = []

for top_percent in [0.10, 0.20, 0.25, 0.30]:
    top_count = int(len(risk_df_v3) * top_percent)

    current_rate = (
        risk_df_v3.nlargest(top_count, "TruePositiveProbability")["Actual"]
        .eq("TruePositive")
        .mean()
    )

    proposed_rate = (
        risk_df_v3.nlargest(top_count, "HybridRiskScore")["Actual"]
        .eq("TruePositive")
        .mean()
    )

    rows.append({
        "Top Percentage": f"Top {int(top_percent*100)}%",
        "Current AI-SIEM": current_rate,
        "Proposed Framework": proposed_rate,
        "Improvement": proposed_rate - current_rate
    })

top_percentage_results = pd.DataFrame(rows)
top_percentage_results

### Cell 11 — Prioritisation Line Chart (Figure 2)

Plots the True Positive Rate for both approaches across all threshold groups.  
A higher line means the approach surfaces more genuine incidents within the reviewed set.

> **Figure saved as:** `figure_prioritisation_comparison.png`

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    top_percentage_results["Top Percentage"],
    top_percentage_results["Current AI-SIEM"],
    marker="o", label="Current AI-SIEM"
)
plt.plot(
    top_percentage_results["Top Percentage"],
    top_percentage_results["Proposed Framework"],
    marker="o", label="Proposed Framework"
)
plt.ylabel("True Positive Rate")
plt.xlabel("Top Review Group")
plt.title("Current AI-SIEM vs Proposed Framework — Alert Prioritisation")
plt.legend()
plt.tight_layout()
plt.savefig("figure_prioritisation_comparison.png", dpi=300)
plt.show()
print("Saved: figure_prioritisation_comparison.png")

---
## Experiment 3 — Investigation Workload Reduction

### Cell 12 — Alert Rows vs Incident Groups

The **Current AI-SIEM** approach presents analysts with individual alert/evidence rows.  
The **Proposed Framework** groups these rows by `IncidentId`, reducing the review count to the number of distinct incidents.

**Workload Reduction %** = `(1 - unique_incidents / total_rows) × 100`

> **Table saved as:** `table_investigation_workload.csv`  
> **Figure saved as:** `figure_investigation_workload.png`

In [ ]:
total_rows = len(raw_df)

if "IncidentId" in raw_df.columns:
    total_incidents = raw_df["IncidentId"].nunique()
    workload_reduction = (1 - total_incidents / total_rows) * 100

    investigation_results = pd.DataFrame([
        {
            "Approach": "Current AI-SIEM",
            "Review Unit": "Individual evidence/alert rows",
            "Review Items": total_rows,
            "Workload Reduction": "Baseline"
        },
        {
            "Approach": "Proposed Framework",
            "Review Unit": "Incident-level grouped view",
            "Review Items": total_incidents,
            "Workload Reduction": f"{workload_reduction:.2f}%"
        }
    ])

investigation_results

### Cell 13 — Investigation Workload Bar Chart (Figure 3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(
    investigation_results["Approach"],
    investigation_results["Review Items"]
)
plt.ylabel("Number of Review Items")
plt.title("Investigation Workload Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figure_investigation_workload.png", dpi=300)
plt.show()
print("Saved: figure_investigation_workload.png")

---
## Experiment 4 — Automated Response Recommendation

### Cell 14 — Playbook Mapping

Maps each alert to a recommended response action using a **category × incident-grade** decision matrix.

**Logic:**
- `TruePositive` → Immediate containment action based on `Category`
- `BenignPositive` → Validate and monitor
- `FalsePositive` → No containment; tune detection rule
- Unknown → Analyst review required

This simulates the automated response recommendation contribution of the proposed framework.

In [ ]:
response_map = {
    "Malware":             "Isolate endpoint; run malware scan; collect forensic evidence",
    "Phishing":            "Disable account; reset password; review MFA and mailbox rules",
    "CredentialAccess":    "Reset credentials; revoke sessions; enforce MFA review",
    "CommandAndControl":   "Block domain/IP; isolate endpoint; investigate process tree",
    "Exfiltration":        "Block outbound channel; isolate host; review accessed data",
    "Reconnaissance":      "Monitor source; block repeated scanning; increase logging",
    "Persistence":         "Remove persistence mechanism; isolate host; review autoruns",
    "PrivilegeEscalation": "Disable affected account; review admin group changes",
    "Execution":           "Isolate endpoint if confirmed; collect process and command-line evidence",
    "InitialAccess":       "Validate access source; reset credentials if account compromise is suspected",
    "LateralMovement":     "Isolate affected hosts; review authentication paths and admin activity"
}

def recommend_response(row):
    category = str(row.get("Category", "Unknown"))
    grade    = str(row.get("IncidentGrade", "Unknown"))
    base     = response_map.get(category, "Analyst review required")

    if grade == "TruePositive":
        return "Immediate action: " + base
    elif grade == "BenignPositive":
        return "Validate and monitor: " + base
    elif grade == "FalsePositive":
        return "No containment; tune detection logic"
    else:
        return "Analyst review required"

raw_df["RecommendedResponse"] = raw_df.apply(recommend_response, axis=1)

# Preview sample
raw_df[["Category", "IncidentGrade", "RecommendedResponse"]].head(20)

### Cell 15 — Response Summary by IncidentGrade

Groups records by `IncidentGrade` and counts total records and distinct response actions issued.

> **Table saved as:** `table_response_summary.csv`  
> **Figure saved as:** `figure_response_summary.png`

In [ ]:
response_summary = raw_df.groupby(["IncidentGrade"]).agg(
    TotalRecords=("IncidentGrade", "count"),
    UniqueResponses=("RecommendedResponse", "nunique")
).reset_index()

response_summary

### Cell 16 — Response Summary Bar Chart (Figure 4)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(response_summary["IncidentGrade"], response_summary["TotalRecords"])
plt.ylabel("Number of Records")
plt.xlabel("Incident Grade")
plt.title("Response Recommendation Volume by Incident Grade")
plt.tight_layout()
plt.savefig("figure_response_summary.png", dpi=300)
plt.show()
print("Saved: figure_response_summary.png")

### Cell 17 — Top 15 Response Actions

Shows the 15 most frequently issued response actions across the dataset.

> **Table saved as:** `table_response_action_distribution.csv`

In [ ]:
response_action_distribution = raw_df["RecommendedResponse"].value_counts().head(15).reset_index()
response_action_distribution.columns = ["Recommended Response", "Count"]

response_action_distribution

---
## Save All Results

### Cell 18 — Export Tables to CSV and Excel

Saves all result tables as individual CSVs and as a single multi-sheet Excel workbook.  
All files are written to the current working directory.

In [ ]:
# Individual CSVs
pd.DataFrame([current_results]).to_csv("table_detection_performance.csv", index=False)
top_percentage_results.to_csv("table_prioritisation_comparison.csv", index=False)
investigation_results.to_csv("table_investigation_workload.csv", index=False)
response_summary.to_csv("table_response_summary.csv", index=False)
response_action_distribution.to_csv("table_response_action_distribution.csv", index=False)

# Combined Excel workbook
with pd.ExcelWriter("SIEM_Framework_Results_Tables.xlsx") as writer:
    pd.DataFrame([current_results]).to_excel(writer, sheet_name="Detection Performance", index=False)
    top_percentage_results.to_excel(writer, sheet_name="Prioritisation", index=False)
    investigation_results.to_excel(writer, sheet_name="Investigation", index=False)
    response_summary.to_excel(writer, sheet_name="Response Summary", index=False)
    response_action_distribution.to_excel(writer, sheet_name="Response Distribution", index=False)

print("All tables saved successfully.")
print("Files written:")
for f in [
    "table_detection_performance.csv",
    "table_prioritisation_comparison.csv",
    "table_investigation_workload.csv",
    "table_response_summary.csv",
    "table_response_action_distribution.csv",
    "SIEM_Framework_Results_Tables.xlsx"
]:
    print(" -", f)